In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import yaml
from tqdm.auto import tqdm


plt.rcParams["figure.dpi"] = 300

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())

THRESHOLDS = CFG["retrieval"]["thresholds"]
METHOD = CFG["vpr"]["method"]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")


PROJECT_ROOT = find_project_root()
RESULT_DIR = PROJECT_ROOT / "results" 
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD
RETRIEVAL_DIR = RESULT_DIR / "retrieval"


if ADAPTER == "none" or ADAPTER == "None":
    EMBEDDING_NAME = METHOD
else:
    EMBEDDING_NAME = f"{METHOD}_{ADAPTER}"


embedding_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"

RETRIEVAL_DIR.mkdir(parents = True, exist_ok = True)

embedding_metadata = pd.read_parquet(metadata_path)
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()
database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)

embeddings = np.load(embedding_path)

database_embeddings = embeddings[database_mask]
query_embeddings = embeddings[query_mask]


retrieval = np.load(RETRIEVAL_DIR / METHOD / f"{EMBEDDING_NAME}_retrieval.npz")
retrieved_indices = retrieval["indices"]
similarities = retrieval["similarities"]


In [ ]:
# harvisine distance
# https://en.wikipedia.org/wiki/Haversine_formula

def haversine_distance(
        lat1,
        lon1,
        lat2,
        lon2
):
    earth_radius = 6_371_000

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    diff_lat = lat2 - lat1
    diff_lon = lon2 - lon1

    a = (np.sin(diff_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(diff_lon / 2) ** 2)

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius * c



In [ ]:
def ground_truth(query_index, query_metadata, database_metadata, threshold = 25.0):
    query = query_metadata.iloc[query_index]

    haversine_distances = haversine_distance(
        query["lat"],
        query["lon"],
        database_metadata["lat"].to_numpy(),
        database_metadata["lon"].to_numpy()
    )
    ground = np.where(haversine_distances <= threshold)[0]

    return ground, haversine_distances

In [ ]:
def recall_k( retrieved_indices, ground):

    return int(np.isin(retrieved_indices, ground).any())

In [ ]:
K_VALUES = [1, 5, 10, 20]

K_MAX = retrieved_indices.shape[1]
assert max(K_VALUES) <= K_MAX, (
    f"Nur {K_MAX} Treffer gespeichert, Recall@{max(K_VALUES)} unmoeglich"
)

db_lat = database_metadata["lat"].to_numpy()
db_lon = database_metadata["lon"].to_numpy()
db_creator = database_metadata["creator_id"].to_numpy()
db_time = database_metadata["captured_at"].to_numpy().astype("int64")

q_pano = query_metadata["is_pano"].to_numpy().astype(bool)
q_creator = query_metadata["creator_id"].to_numpy()
q_time = query_metadata["captured_at"].to_numpy().astype("int64")

MIN_DAYS_APART = 180.0


def evaluate(label, query_filter=None, gt_filter=None):
    """
    query_filter : bool-Array ueber Queries -- welche Queries zaehlen mit
    gt_filter    : Funktion(qi) -> bool-Array ueber Database, zusaetzliche
                   Bedingung dafuer, dass ein DB-Bild als Treffer zaehlt
    """
    n_localizable = {t: 0 for t in THRESHOLDS}
    hits = {(t, k): 0 for t in THRESHOLDS for k in K_VALUES}
    n_queries = 0

    for qi in tqdm(range(len(query_metadata)), desc=label, leave=False):
        if query_filter is not None and not query_filter[qi]:
            continue
        n_queries += 1

        q = query_metadata.iloc[qi]
        d = haversine_distance(q["lat"], q["lon"], db_lat, db_lon)
        ok = gt_filter(qi) if gt_filter is not None else None
        top = retrieved_indices[qi]

        for t in THRESHOLDS:
            valid = d <= t
            if ok is not None:
                valid = valid & ok
            ground = np.flatnonzero(valid)
            if len(ground) == 0:
                continue
            n_localizable[t] += 1
            for k in K_VALUES:
                if np.isin(top[:k], ground).any():
                    hits[(t, k)] += 1

    print(f"\n{label}   (Queries: {n_queries:,})")
    print(
        f"{'Schwelle':>10} {'loesbar':>10} {'Anteil':>8} "
        + " ".join(f"R@{k:<5}" for k in K_VALUES)
    )
    for t in THRESHOLDS:
        n = n_localizable[t]
        frac = n / n_queries * 100 if n_queries else 0.0
        vals = " ".join(
            f"{hits[(t, k)] / n:<7.3f}" if n else f"{'-':<7}" for k in K_VALUES
        )
        print(f"{t:>8} m {n:>10,} {frac:>7.1f}% {vals}")


# --- 1. Standard --------------------------------------------------------
evaluate("Alle Queries")

# --- 2. Ohne Panorama-Queries (4.1) -------------------------------------
if q_pano.any():
    evaluate("Nur Nicht-Panorama-Queries", query_filter=~q_pano)
else:
    print("\nKeine Panorama-Queries im Datensatz -- Aufteilung entfaellt.")


# --- 3. Zeit-/Creator-disjunkt (4.3) ------------------------------------
def disjoint(qi):
    dt_days = np.abs(q_time[qi] - db_time) / 86_400_000.0
    return (db_creator != q_creator[qi]) | (dt_days > MIN_DAYS_APART)


evaluate(
    f"Hard: anderer creator_id ODER > {MIN_DAYS_APART:g} Tage Abstand",
    gt_filter=disjoint,
)
